# Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Load Dataset

In [ ]:
# Dataset menggunakan delimiter ';' dan baris pertama adalah judul dataset
data = pd.read_csv(
    "gsar_a_2521295_sm1572.csv",
    sep=";",
    skiprows=1,
    encoding="utf-8-sig"
)

# Buang baris dengan SMILES kosong (jika ada) & pastikan label integer
data = data.dropna(subset=["SMILES", "label"]).reset_index(drop=True)
data["label"] = data["label"].astype(int)

print(data.shape)
data[["compound ID", "SMILES", "label"]].head()

# Tokenisasi SMILES


In [ ]:
X_smiles = data["SMILES"].values

# Bangun charset dari SELURUH SMILES di dataset + token start '!' dan end 'E'
charset = set("".join(list(X_smiles)) + "!E")
char_to_int = dict((c, i) for i, c in enumerate(charset))
int_to_char = dict((i, c) for i, c in enumerate(charset))

embed = max([len(smile) for smile in X_smiles]) + 5
vocab_size = len(charset)

print("Jumlah karakter unik (vocab_size):", vocab_size)
print("Panjang embedding (embed):", embed)
print(str(charset))

In [ ]:
char_to_int

# Fungsi Vectorize

In [ ]:
def vectorize(smiles, charset, char_to_int, embed):
    one_hot = np.zeros((smiles.shape[0], embed, len(charset)), dtype=np.int8)
    for i, smile in enumerate(smiles):
        # encode start char
        one_hot[i, 0, char_to_int["!"]] = 1
        # encode karakter SMILES
        for j, c in enumerate(smile):
            if c in char_to_int:
                one_hot[i, j + 1, char_to_int[c]] = 1
        # encode end char (padding sisa)
        one_hot[i, len(smile) + 1:, char_to_int["E"]] = 1
    return one_hot[:, 0:-1, :]

# Vectorize SELURUH dataset (tanpa split)
X_onehot = vectorize(X_smiles, charset, char_to_int, embed)

# Ubah one-hot menjadi integer sequence (siap untuk Embedding layer nantinya)
X_encoded = np.argmax(X_onehot, axis=2)

# Visual Hasil Encoding SMILES

In [ ]:
print("vocab_size:", vocab_size, "| embed:", embed)
print("X_onehot shape :", X_onehot.shape)   # (jumlah_senyawa, panjang_sequence, jumlah_karakter_unik)
print("X_encoded shape:", X_encoded.shape)  # (jumlah_senyawa, panjang_sequence)

idx = 0  # ganti index sesuai molekul yang ingin dilihat

print("\nSMILES asli:", X_smiles[idx])
print()
print("One-hot encoding, 10 posisi pertama:")
print(X_onehot[idx][:10])

print("Integer-encoded, 20 posisi pertama:")
print(X_encoded[idx][:20])

decoded = "".join([int_to_char[i] for i in X_encoded[idx]])
print("Hasil decode balik:", decoded)

# Penggabungan Hasil Encoding SMILES dengan Label

In [ ]:
y = data["label"].values

print("X_encoded shape:", X_encoded.shape)
print("y shape         :", y.shape)

# Validasi jumlah baris sudah sinkron antara fitur dan label
assert X_encoded.shape[0] == y.shape[0], "Jumlah baris SMILES dan label tidak sinkron!"

# Split Train/Test

In [ ]:
X = X_encoded          # hasil tahap tokenisasi sebelumnya, shape (846, seq_len)
y_arr = y.reshape(-1, 1) if y.ndim == 1 else y

X_train, X_test, Y_train, Y_test = train_test_split(
    X, y_arr,
    test_size=0.3,
    random_state=42,
    stratify=y_arr
)

print(X_train.shape, X_test.shape)

# Dataset & Dataset Loader

In [ ]:
class SmilesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = SmilesDataset(X_train, Y_train)
test_dataset  = SmilesDataset(X_test, Y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Pembentukan model Baseline

In [ ]:
class LSTMBaseline(nn.Module):
    def __init__(self, vocab_size, embedding_dim=50, units_list=[64]):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm_layers = nn.ModuleList()
        input_size = embedding_dim
        for units in units_list:
            self.lstm_layers.append(nn.LSTM(input_size=input_size, hidden_size=units, batch_first=True))
            input_size = units
        self.output_layer = nn.Linear(units_list[-1], 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out = self.embedding(x)
        for lstm in self.lstm_layers:
            out, _ = lstm(out)
        out = out[:, -1, :]
        out = self.output_layer(out)
        return self.sigmoid(out)


def train_model(model, train_loader, test_loader, epochs=50, lr=0.001, device=device):
    model.to(device)
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * xb.size(0)
            train_correct += ((preds > 0.5).float() == yb).sum().item()
            train_total += xb.size(0)

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb)
                loss = criterion(preds, yb)
                val_loss += loss.item() * xb.size(0)
                val_correct += ((preds > 0.5).float() == yb).sum().item()
                val_total += xb.size(0)

        history["loss"].append(train_loss / train_total)
        history["accuracy"].append(train_correct / train_total)
        history["val_loss"].append(val_loss / val_total)
        history["val_accuracy"].append(val_correct / val_total)

        print(f"Epoch {epoch+1}/{epochs} - loss: {history['loss'][-1]:.4f} - acc: {history['accuracy'][-1]:.4f} "
              f"- val_loss: {history['val_loss'][-1]:.4f} - val_acc: {history['val_accuracy'][-1]:.4f}")

    return history


def get_predictions(model, data_loader, device=device, threshold=0.5):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for xb, yb in data_loader:
            xb = xb.to(device)
            probs = model(xb).cpu().numpy().flatten()
            preds = (probs >= threshold).astype(int)
            y_true.extend(yb.numpy().flatten().astype(int))
            y_pred.extend(preds)
    return np.array(y_true), np.array(y_pred)


def compute_confusion_matrix(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    return TP, TN, FP, FN


def compute_metrics(TP, TN, FP, FN, eps=1e-10):
    accuracy  = (TP + TN) / (TP + TN + FP + FN + eps)
    recall    = TP / (TP + FN + eps)
    precision = TP / (TP + FP + eps)
    f1_score  = 2 * (precision * recall) / (precision + recall + eps)
    return {"accuracy": accuracy, "recall": recall, "precision": precision, "f1_score": f1_score}


def print_confusion_matrix(TP, TN, FP, FN, model_name=""):
    cm_df = pd.DataFrame(
        [[TP, FN], [FP, TN]],
        index=["Aktual Aktif (1)", "Aktual Inaktif (0)"],
        columns=["Prediksi Aktif (1)", "Prediksi Inaktif (0)"]
    )
    print(f"\nConfusion Matrix - {model_name}")
    print(cm_df)
    return cm_df

# Baseline 1

In [ ]:
print("="*60)
print("BASELINE 1 | LSTM layers: [64]")
print("="*60)

model_b1 = LSTMBaseline(vocab_size=vocab_size, embedding_dim=50, units_list=[64])
print(model_b1)

history_b1 = train_model(model_b1, train_loader, test_loader, epochs=50, lr=0.001)

In [ ]:
y_true_b1, y_pred_b1 = get_predictions(model_b1, test_loader)
TP1, TN1, FP1, FN1 = compute_confusion_matrix(y_true_b1, y_pred_b1)
metrics_b1 = compute_metrics(TP1, TN1, FP1, FN1)

print_confusion_matrix(TP1, TN1, FP1, FN1, model_name="Baseline 1")
print(f"\nAccuracy  : {metrics_b1['accuracy']:.4f}")
print(f"Recall    : {metrics_b1['recall']:.4f}")
print(f"Precision : {metrics_b1['precision']:.4f}")
print(f"F1-Score  : {metrics_b1['f1_score']:.4f}")

# BAseline 2

In [ ]:
print("="*60)
print("BASELINE 2 | LSTM layers: [64, 128]")
print("="*60)

model_b2 = LSTMBaseline(vocab_size=vocab_size, embedding_dim=50, units_list=[64, 128])
print(model_b2)

history_b2 = train_model(model_b2, train_loader, test_loader, epochs=50, lr=0.001)

In [ ]:
y_true_b2, y_pred_b2 = get_predictions(model_b2, test_loader)
TP2, TN2, FP2, FN2 = compute_confusion_matrix(y_true_b2, y_pred_b2)
metrics_b2 = compute_metrics(TP2, TN2, FP2, FN2)

print_confusion_matrix(TP2, TN2, FP2, FN2, model_name="Baseline 2")
print(f"\nAccuracy  : {metrics_b2['accuracy']:.4f}")
print(f"Recall    : {metrics_b2['recall']:.4f}")
print(f"Precision : {metrics_b2['precision']:.4f}")
print(f"F1-Score  : {metrics_b2['f1_score']:.4f}")

# Baseline 3

In [ ]:
print("="*60)
print("BASELINE 3 | LSTM layers: [64, 128, 256]")
print("="*60)

model_b3 = LSTMBaseline(vocab_size=vocab_size, embedding_dim=50, units_list=[64, 128, 256])
print(model_b3)

history_b3 = train_model(model_b3, train_loader, test_loader, epochs=50, lr=0.001)

In [ ]:
y_true_b3, y_pred_b3 = get_predictions(model_b3, test_loader)
TP3, TN3, FP3, FN3 = compute_confusion_matrix(y_true_b3, y_pred_b3)
metrics_b3 = compute_metrics(TP3, TN3, FP3, FN3)

print_confusion_matrix(TP3, TN3, FP3, FN3, model_name="Baseline 3")
print(f"\nAccuracy  : {metrics_b3['accuracy']:.4f}")
print(f"Recall    : {metrics_b3['recall']:.4f}")
print(f"Precision : {metrics_b3['precision']:.4f}")
print(f"F1-Score  : {metrics_b3['f1_score']:.4f}")

# Penggabungan Hasil ke 1 Dataframe

In [ ]:
evaluation_results = {
    "Baseline 1": {"TP": TP1, "TN": TN1, "FP": FP1, "FN": FN1, **metrics_b1},
    "Baseline 2": {"TP": TP2, "TN": TN2, "FP": FP2, "FN": FN2, **metrics_b2},
    "Baseline 3": {"TP": TP3, "TN": TN3, "FP": FP3, "FN": FN3, **metrics_b3},
}

summary_df = pd.DataFrame(evaluation_results).T
summary_df = summary_df[["TP", "TN", "FP", "FN", "accuracy", "recall", "precision", "f1_score"]]
summary_df

In [ ]:
# Ringkasan histori training (loss & accuracy epoch terakhir) untuk ketiga baseline
histories = {"Baseline 1": history_b1, "Baseline 2": history_b2, "Baseline 3": history_b3}

training_summary_df = pd.DataFrame({
    name: {
        "train_acc": hist["accuracy"][-1],
        "val_acc": hist["val_accuracy"][-1],
        "train_loss": hist["loss"][-1],
        "val_loss": hist["val_loss"][-1],
    }
    for name, hist in histories.items()
}).T

training_summary_df